# Predict all crops with the saved XGBoost pipeline

This local Jupyter notebook uses the tuned XGBoost pipeline by default. Set `MODEL_VARIANT` to `original` to compare with the retained original pipeline. It loads `future_horizon_features.csv`, predicts every crop/location/H1-H12 row, and saves variant-specific forecasts under `artifacts/predictions`. It never creates hypothetical features or replaces missing future context with zero.

In [1]:
# Run this only if the local Jupyter environment is missing dependencies.
# %pip install -r ../../requirements.txt

## Configure local project files

The model is read from `artifacts/demand/models`, future features from `data/processed/demand`, and predictions are written to `artifacts/demand/predictions`. The project root is detected whether Jupyter starts in the project root or the `notebooks` folder.

In [2]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

def find_project_root(start_path):
    start_path = start_path.resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "pipeline_config.json").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Start Jupyter inside the demand-forecasting project."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
MODEL_VARIANT = "tuned"  # Change to "original" to use the retained original model.
MODEL_FILES = {
    "tuned": "global_xgboost_tuned_pipeline.joblib",
    "original": "global_xgboost_pipeline.joblib",
}
if MODEL_VARIANT not in MODEL_FILES:
    raise ValueError(f"MODEL_VARIANT must be one of {list(MODEL_FILES)}")
MODEL_FILE = PROJECT_ROOT / "artifacts" / "demand" / "models" / MODEL_FILES[MODEL_VARIANT]
FUTURE_DATA_FILE = PROJECT_ROOT / "data" / "processed" / "demand" / "future_horizon_features.csv"
METADATA_FILE = PROJECT_ROOT / "artifacts" / "demand" / "reports" / (
    "xgboost_tuning_metadata.json" if MODEL_VARIANT == "tuned"
    else "xgboost_training_metadata.json"
)
OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "demand" / "predictions"

for required_file in [MODEL_FILE, FUTURE_DATA_FILE, METADATA_FILE]:
    if not required_file.exists():
        raise FileNotFoundError(f"Required file not found: {required_file}")

training_metadata = json.loads(METADATA_FILE.read_text(encoding="utf-8"))
trained_sklearn_version = training_metadata.get("versions", {}).get("scikit_learn")
if trained_sklearn_version and sklearn.__version__ != trained_sklearn_version:
    raise RuntimeError(
        f"scikit-learn version mismatch: model={trained_sklearn_version}, "
        f"local={sklearn.__version__}. Run `%pip install scikit-learn=={trained_sklearn_version}` "
        "and then restart the Jupyter kernel."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Selected model: {MODEL_VARIANT} ({MODEL_FILE})")
print(f"Future features: {FUTURE_DATA_FILE}")

Project root: C:\demand-forecasting
Model: C:\demand-forecasting\artifacts\models\global_xgboost_pipeline.joblib
Future features: C:\demand-forecasting\data\processed\future_horizon_features.csv


## Load and validate future features

Prediction stops when required columns or future holiday values are missing. This prevents plausible-looking forecasts based on invented context.

In [3]:
try:
    model_pipeline = joblib.load(MODEL_FILE)
except (AttributeError, ModuleNotFoundError) as error:
    raise RuntimeError(
        "Could not load the pipeline. Use the same scikit-learn/XGBoost versions "
        "as the training environment, then restart the Jupyter kernel."
    ) from error
future_data = pd.read_csv(
    FUTURE_DATA_FILE,
    parse_dates=["forecast_origin_month", "target_month"],
)

feature_columns = list(model_pipeline.feature_names_in_)
missing_features = [column for column in feature_columns if column not in future_data.columns]
if missing_features:
    raise ValueError(f"Future dataset is missing model features: {missing_features}")

required_future_context = [
    "target_public_holiday_count",
    "target_cny",
    "target_hari_raya",
    "target_gawai",
    "target_christmas",
    "target_major_festival",
    "target_month_normal_temperature",
    "target_month_normal_precipitation",
]
missing_context = future_data[required_future_context].isna().sum()
missing_context = missing_context[missing_context > 0]
if not missing_context.empty:
    raise ValueError(
        "Future context is incomplete. Add verified future holiday data and regenerate "
        "future_horizon_features.csv. Missing values: " + str(missing_context.to_dict())
    )

expected_rows = future_data[["crop", "location"]].drop_duplicates().shape[0] * 12
if len(future_data) != expected_rows:
    raise ValueError(f"Expected {expected_rows} H1-H12 rows but found {len(future_data)}.")

print(f"Validated {len(future_data):,} prediction rows.")
print(f"Target range: {future_data.target_month.min().date()} to {future_data.target_month.max().date()}")

Validated 540 prediction rows.
Target range: 2026-01-01 to 2026-12-01


## Predict all crop, location, and horizon rows

In [4]:
X_future = future_data[feature_columns].copy()
predicted_demand = model_pipeline.predict(X_future)

future_data["predicted_demand_kg"] = np.maximum(predicted_demand, 0)
forecast_output = future_data[
    [
        "forecast_origin_month",
        "target_month",
        "forecast_horizon",
        "crop",
        "location",
        "predicted_demand_kg",
    ]
].sort_values(["target_month", "crop", "location"]).reset_index(drop=True)

display(forecast_output.head(30))

,forecast_origin_month,target_month,forecast_horizon,crop,location,predicted_demand_kg
0,2025-12-01,2026-01-01,1,Bittergourd,Kuching,2578.749756
1,2025-12-01,2026-01-01,1,Bittergourd,Miri,1747.337524
2,2025-12-01,2026-01-01,1,Bittergourd,Sibu,2078.118408
3,2025-12-01,2026-01-01,1,Cabbage,Kuching,5398.724609
4,2025-12-01,2026-01-01,1,Cabbage,Miri,3092.419922
5,2025-12-01,2026-01-01,1,Cabbage,Sibu,3357.790039
6,2025-12-01,2026-01-01,1,Choysum,Kuching,3473.353516
7,2025-12-01,2026-01-01,1,Choysum,Miri,2453.032227
8,2025-12-01,2026-01-01,1,Choysum,Sibu,2830.062744
9,2025-12-01,2026-01-01,1,Cucumber,Kuching,4882.946777


## Select a target date range

The example selects October through November 2026. Change the dates without changing model features.

In [5]:
SELECTED_START = pd.Timestamp("2026-10-01")
SELECTED_END = pd.Timestamp("2026-11-01")

selected_forecast = forecast_output[
    forecast_output["target_month"].between(SELECTED_START, SELECTED_END)
].copy()

if selected_forecast.empty:
    raise ValueError("No forecast rows fall within the selected target range.")

print(f"Selected rows: {len(selected_forecast):,}")
display(selected_forecast)

Selected rows: 90


,forecast_origin_month,target_month,forecast_horizon,crop,location,predicted_demand_kg
405,2025-12-01,2026-10-01,10,Bittergourd,Kuching,1536.802856
406,2025-12-01,2026-10-01,10,Bittergourd,Miri,1417.589355
407,2025-12-01,2026-10-01,10,Bittergourd,Sibu,1395.946655
408,2025-12-01,2026-10-01,10,Cabbage,Kuching,3411.828369
409,2025-12-01,2026-10-01,10,Cabbage,Miri,2311.424805
...,...,...,...,...,...,...
490,2025-12-01,2026-11-01,11,Sweetcorn,Miri,3322.403564
491,2025-12-01,2026-11-01,11,Sweetcorn,Sibu,3338.799561
492,2025-12-01,2026-11-01,11,Tomato,Kuching,3780.696777
493,2025-12-01,2026-11-01,11,Tomato,Miri,2509.954834


## Save full and selected forecasts

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

full_name = f"xgboost_{MODEL_VARIANT}_future_forecast_h1_h12.csv"
selected_name = (
    f"xgboost_{MODEL_VARIANT}_forecast_"
    f"{SELECTED_START:%Y_%m}_to_{SELECTED_END:%Y_%m}.csv"
)

forecast_output.to_csv(OUTPUT_DIR / full_name, index=False)
selected_forecast.to_csv(OUTPUT_DIR / selected_name, index=False)

print(f"Full forecast: {OUTPUT_DIR / full_name}")
print(f"Selected forecast: {OUTPUT_DIR / selected_name}")

Full forecast: C:\demand-forecasting\artifacts\predictions\xgboost_future_forecast_h1_h12.csv
Selected forecast: C:\demand-forecasting\artifacts\predictions\xgboost_forecast_2026_10_to_2026_11.csv
